# RealThor Evaluation Suite — Solution

We have three prompt versions:

- **v1** — original, no explicit structure guidance
- **v2** — adds structure but is overly conservative
- **v3** — structured with a calibrated decision framework

We run all three against the 20-scenario golden dataset and use a decision gate to determine which version to ship.

In [1]:
import os
import pandas as pd
from typing import Literal
from pydantic import BaseModel
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [2]:
load_dotenv()

True

In [3]:
with open("../prompts/system_prompt_v1.md") as f:
    PROMPT_V1 = f.read()

with open("../prompts/system_prompt_v2.md") as f:
    PROMPT_V2 = f.read()

with open("../prompts/system_prompt_v3.md") as f:
    PROMPT_V3 = f.read()

In [4]:
demand = pd.read_csv("../data/demand_signals.csv")

In [5]:
supply = pd.read_csv("../data/supply_snapshot.csv")

In [6]:
scenarios = pd.read_csv("eval_golden_dataset.csv")

## Output Schema and Structured LLM

In [7]:
class ListingRecommendation(BaseModel):
    """Structured recommendation from RealThor."""
    priority: Literal["high", "medium", "low"]
    demand_reasoning: str
    supply_reasoning: str
    recommended_action: Literal["prioritize_new_listings", "monitor_opportunity", "deprioritize"]
    confidence: Literal["high", "medium", "low"]

In [8]:
provider = os.getenv("LLM_PROVIDER", "openai")
model_kwargs = {"model": "gpt-4o-mini", "temperature": 0.0}
if provider == "vocareum":
    model_kwargs |= {"base_url": "https://openai.vocareum.com/v1", "api_key": os.getenv("VOCAREUM_API_KEY")}
llm = ChatOpenAI(**model_kwargs)
structured_llm = llm.with_structured_output(ListingRecommendation)

## Helper Functions

In [9]:
def build_context(region: str, property_type: str, bedrooms: int) -> str:
    d = demand[
        (demand["region"] == region)
        & (demand["property_type"] == property_type)
        & (demand["bedrooms"] == bedrooms)
    ]
    s = supply[
        (supply["region"] == region)
        & (supply["property_type"] == property_type)
        & (supply["bedrooms"] == bedrooms)
    ]
    if d.empty or s.empty:
        return "No data available for the requested segment."
    return (
        f"Region: {region}\n"
        f"Property: {bedrooms}-bedroom {property_type}\n\n"
        f"Demand signals:\n{d.to_string(index=False)}\n\n"
        f"Supply snapshot:\n{s.to_string(index=False)}"
    )

In [10]:
def ask_realthor(region: str, property_type: str, bedrooms: int, prompt: str) -> ListingRecommendation:
    context = build_context(region, property_type, bedrooms)
    return structured_llm.invoke([
        SystemMessage(content=prompt),
        HumanMessage(content=context),
    ])

In [11]:
def evaluate_output(result: ListingRecommendation, row) -> dict:
    priority_match = result.priority == row["expected_priority"]
    action_match = result.recommended_action == row["expected_action"]
    passed = priority_match and action_match
    failure_reason = ""
    if not priority_match:
        failure_reason += f"priority: got '{result.priority}', expected '{row['expected_priority']}'; "
    if not action_match:
        failure_reason += f"action: got '{result.recommended_action}', expected '{row['expected_action']}'"
    return {
        "passed": passed,
        "priority": result.priority,
        "action": result.recommended_action,
        "failure_reason": failure_reason.strip("; "),
    }

## Run Evaluation — All Three Versions

In [12]:
all_results = []

for version, prompt in [("v1", PROMPT_V1), ("v2", PROMPT_V2), ("v3", PROMPT_V3)]:
    for _, row in scenarios.iterrows():
        result = ask_realthor(row["region"], row["property_type"], row["bedrooms"], prompt)
        eval_result = evaluate_output(result, row)
        all_results.append({
            "scenario_id": row["scenario_id"],
            "prompt_version": version,
            **eval_result,
        })

In [13]:
results_df = pd.DataFrame(all_results)
display(results_df)

,scenario_id,prompt_version,passed,priority,action,failure_reason
0,G001,v1,True,high,prioritize_new_listings,
1,G002,v1,False,high,prioritize_new_listings,"priority: got 'high', expected 'low'; action: ..."
2,G003,v1,False,high,prioritize_new_listings,"priority: got 'high', expected 'medium'; actio..."
3,G004,v1,True,high,prioritize_new_listings,
4,G005,v1,False,medium,monitor_opportunity,"priority: got 'medium', expected 'low'; action..."
5,G006,v1,True,high,prioritize_new_listings,
6,G007,v1,False,medium,monitor_opportunity,"priority: got 'medium', expected 'low'; action..."
7,G008,v1,True,high,prioritize_new_listings,
8,G009,v1,False,high,prioritize_new_listings,"priority: got 'high', expected 'low'; action: ..."
9,G010,v1,False,high,prioritize_new_listings,"priority: got 'high', expected 'medium'; actio..."


## Comparison Table

In [14]:
summary = results_df.groupby("prompt_version")["passed"].agg(
    total="count",
    passed="sum"
)
summary["pass_rate"] = summary["passed"] / summary["total"]
display(summary)

,total,passed,pass_rate
prompt_version,,,
v1,20,6,0.30
v2,20,9,0.45
v3,20,19,0.95


## Regression Analysis

A regression is a scenario that passed in v1 but fails in the candidate version.

In [15]:
v1_passing = set(
    results_df[(results_df["prompt_version"] == "v1") & results_df["passed"]]["scenario_id"]
)

In [16]:
def find_regressions(version: str) -> set:
    version_failing = set(
        results_df[(results_df["prompt_version"] == version) & ~results_df["passed"]]["scenario_id"]
    )
    return v1_passing & version_failing

In [17]:
regressions_v2 = find_regressions("v2")
regressions_v3 = find_regressions("v3")

print(f"v2 regressions vs v1: {regressions_v2 or 'none'}")
print(f"v3 regressions vs v1: {regressions_v3 or 'none'}")

v2 regressions vs v1: {'G008', 'G006', 'G004', 'G017'}
v3 regressions vs v1: none


## Decision Gate

A candidate version ships only if it meets **both** conditions:
1. **No regressions** — no scenario that passed in v1 fails in the candidate
2. **Meaningful improvement** — pass rate is at least 10 percentage points above v1

In [18]:
v1_rate = summary.loc["v1", "pass_rate"]
v2_rate = summary.loc["v2", "pass_rate"]
v3_rate = summary.loc["v3", "pass_rate"]

In [19]:
accept_v2 = v2_rate >= v1_rate + 0.10 and len(regressions_v2) == 0
accept_v3 = v3_rate >= v1_rate + 0.10 and len(regressions_v3) == 0

print(f"v1 pass rate: {v1_rate:.0%}")
print(f"v2 pass rate: {v2_rate:.0%} | regressions: {len(regressions_v2)} | Decision: {'ACCEPT' if accept_v2 else 'REJECT'}")
print(f"v3 pass rate: {v3_rate:.0%} | regressions: {len(regressions_v3)} | Decision: {'ACCEPT' if accept_v3 else 'REJECT'}")
print()

if accept_v3:
    print("Ship v3.")
elif accept_v2:
    print("Ship v2 — but v3 is preferred if it recovers.")
else:
    print("Do not ship. Keep v1.")

v1 pass rate: 30%
v2 pass rate: 45% | regressions: 4 | Decision: REJECT
v3 pass rate: 95% | regressions: 0 | Decision: ACCEPT

Ship v3.
